In [ ]:
# --- CELL 1 : Imports and Environment Setup ---

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.options.plotting.backend = "plotly"

TEMPLATE = "plotly_white"

# --- Path Configuration ---
CURRENT_PATH = Path.cwd()
if CURRENT_PATH.name == 'notebooks':
    PROJECT_ROOT = CURRENT_PATH.parent
else:
    PROJECT_ROOT = CURRENT_PATH

DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "reports" / "figures"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Directory: {DATA_DIR}")

In [ ]:
# --- CELL 2: Data Loading, Consolidation & Delta-T Computation ---

try:
    part_files = list(DATA_DIR.glob("*_processed.parquet"))
    
    if not part_files:
        print("No partial files found. Loading telemetry_full.parquet...")
        df = pd.read_parquet(DATA_DIR / "telemetry_full.parquet")
    else:
        print(f"Found {len(part_files)} processed files. Consolidating...")
        for f in part_files:
            print(f"  -> {f.name}")
        df = pd.concat([pd.read_parquet(f) for f in part_files], ignore_index=True)

    # --- Delta-T computation ---
    print("\nComputing delta_t...")
    df.sort_values(by=['tractor', 'source_file', 'timestamp'], inplace=True)
    df['delta_t'] = df.groupby(['tractor', 'source_file'])['timestamp'].diff().fillna(0)

    # Gaps > 300 s (5 min) → engine off / logger stopped → zero out
    df.loc[df['delta_t'] > 300, 'delta_t'] = 0
    df.loc[df['delta_t'] < 0, 'delta_t'] = 0  # guard against sort artifacts

    # Instantaneous fuel consumed per row (L)
    df['liters_consumed'] = (df['fuel_rate'] * df['delta_t']) / 3600.0

    # --- Report ---
    total_recs = len(df)
    total_hours = df['delta_t'].sum() / 3600

    print("-" * 50)
    print(f"Load complete!")
    print(f"Total records : {total_recs:,.0f}")
    print(f"Total hours   : {total_hours:,.2f} h")
    print("-" * 50)

    if total_recs < 30_000_000:
        print("⚠️ Under 30M records — verify all *_processed.parquet files are present.")

except Exception as e:
    print(f"CRITICAL ERROR: {e}")

In [ ]:
# --- CELL 3: Fleet Operational Summary ---

summary = df.groupby('tractor').agg({
    'delta_t': 'sum',
    'liters_consumed': 'sum',
    'source_file': 'nunique',
    'veh_speed': 'mean',
    'fuel_rate': 'mean',
}).reset_index()

summary['Total Hours (h)'] = (summary['delta_t'] / 3600).round(2)
summary['Total Diesel (L)'] = summary['liters_consumed'].round(2)
summary['Avg Consumption (L/h)'] = (summary['Total Diesel (L)'] / summary['Total Hours (h)']).round(2)

final_table = summary[['tractor', 'source_file', 'Total Hours (h)', 'Total Diesel (L)', 'Avg Consumption (L/h)']].copy()
final_table.columns = ['Tractor', 'Missions', 'Operational Hours', 'Total Consumption (L)', 'Average (L/h)']

print("=== FLEET OPERATIONAL SUMMARY ===")
display(final_table.style.background_gradient(cmap='Blues', subset=['Operational Hours', 'Total Consumption (L)']))

total_h = summary['Total Hours (h)'].sum()
total_l = summary['Total Diesel (L)'].sum()
print(f"\nFleet total: {total_h:,.2f} h | {total_l:,.2f} L consumed")

In [ ]:
# --- CELL 4 (optional): List unique activity labels ---

unique_activities = sorted(df['activity'].unique())
print("Unique activities found:\n")
for act in unique_activities:
    print(f"  '{act}'")

In [ ]:
# --- CELL 5: Activity Distribution (Treemap) ---

# Aggregate hours & fuel by tractor × activity
tree_data = df.groupby(['tractor', 'activity'])[['delta_t', 'liters_consumed']].sum().reset_index()
tree_data['Hours'] = tree_data['delta_t'] / 3600

# Clean up the one misleading label
tree_data['activity_label'] = tree_data['activity'].replace({'not working': 'Idle'})

# Treemap: size = hours, color = total fuel
fig = px.treemap(
    tree_data,
    path=[px.Constant("Fendt Fleet"), 'tractor', 'activity_label'],
    values='Hours',
    color='liters_consumed',
    color_continuous_scale='RdBu_r',
    title="<b>Season Time Distribution:</b> Size = Hours, Color = Total Consumption (L)",
    hover_data={'Hours': ':.2f', 'liters_consumed': ':.2f'},
)

fig.update_layout(
    template=TEMPLATE,
    margin=dict(b=40, l=20, r=20, t=50),
)
fig.show()

In [ ]:
# --- CELL 6: Operational Profile — Speed & RPM Histograms ---

print("Generating operational profile histograms...")

target_df = full_df if 'full_df' in locals() else df

SAMPLE_N = 100_000
sample_df = target_df.sample(n=SAMPLE_N, random_state=42) if len(target_df) > SAMPLE_N else target_df.copy()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("<b>Speed Distribution (m/s)</b>", "<b>RPM Distribution</b>"),
)

tractors = sample_df['tractor'].unique()
palette = px.colors.qualitative.Bold

for i, tractor in enumerate(tractors):
    subset = sample_df[sample_df['tractor'] == tractor]
    color = palette[i % len(palette)]

    # Speed
    speed = subset.query('0.1 < veh_speed < 15')['veh_speed']
    fig.add_trace(go.Histogram(
        x=speed, name=tractor, nbinsx=80, histnorm='percent',
        opacity=0.6, marker_color=color, legendgroup=tractor,
    ), row=1, col=1)

    # RPM
    rpm = subset.query('engine_rpm > 100')['engine_rpm']
    fig.add_trace(go.Histogram(
        x=rpm, name=tractor, nbinsx=80, histnorm='percent',
        opacity=0.6, marker_color=color, legendgroup=tractor, showlegend=False,
    ), row=1, col=2)

# Idle thresholds
fig.add_vline(x=1.0, line_width=2, line_dash="dash", line_color="black",
              annotation_text="Idle cutoff", annotation_position="top right", row=1, col=1)
fig.add_vline(x=1000, line_width=2, line_dash="dash", line_color="black",
              annotation_text="Idle cutoff", annotation_position="top right", row=1, col=2)

fig.update_layout(
    title_text="<b>Operational Profile</b>",
    barmode='overlay', height=500, template=TEMPLATE,
    xaxis_title="Speed (m/s)", yaxis_title="Frequency (%)",
)
fig.update_xaxes(title_text="Engine Speed (RPM)", row=1, col=2)
fig.update_yaxes(title_text="Frequency (%)", row=1, col=2)
fig.show()

In [ ]:
# --- CELL 7: Efficiency Map — RPM vs Fuel Rate ---

scatter_sample = df.sample(n=50_000, random_state=22)

fig = px.scatter(
    scatter_sample,
    x='engine_rpm', y='fuel_rate', color='tractor',
    size='veh_speed', size_max=12, opacity=0.6,
    title="<b>Efficiency Map:</b> RPM vs Fuel Rate (bubble size = speed)",
    labels={'engine_rpm': 'Engine Speed (RPM)', 'fuel_rate': 'Fuel Rate (L/h)'},
    hover_data=['activity'],
)
fig.update_layout(template=TEMPLATE)
fig.show()

In [ ]:
# --- CELL 8: Event-Based State Classification (Off / Idle / Working) ---

print("--- Applying event-based state classification ---")

# 1. Thresholds
SPEED_THRESHOLD = 0.75       # m/s (~2.7 km/h)
RPM_ON_THRESHOLD = 100       # below → engine off
PTO_ACTIVE_THRESHOLD = 150   # RPM → PTO engaged
MIN_IDLE_DURATION = 0        # seconds (tune to filter short pauses)

# 2. Instantaneous classification
mask_off = df['engine_rpm'] < RPM_ON_THRESHOLD
mask_low_speed = df['veh_speed'] < SPEED_THRESHOLD
mask_pto_off = df['pto_rpm'].fillna(0) < PTO_ACTIVE_THRESHOLD

df['state_raw'] = 'Working'
df.loc[mask_off, 'state_raw'] = 'Off'
df.loc[(~mask_off) & mask_low_speed & mask_pto_off, 'state_raw'] = 'Idle_Candidate'

# 3. Event grouping — assign unique ID per contiguous state block
df['event_id'] = (df['state_raw'] != df['state_raw'].shift()).cumsum()

# 4. Duration filter — short idle candidates → reclassify as Working (maneuvers)
event_durations = df.groupby('event_id')['delta_t'].transform('sum')

df['state'] = df['state_raw']
df.loc[(df['state_raw'] == 'Idle_Candidate') & (event_durations < MIN_IDLE_DURATION), 'state'] = 'Working'
df.loc[(df['state_raw'] == 'Idle_Candidate') & (event_durations >= MIN_IDLE_DURATION), 'state'] = 'Idle'

# 5. Idle fuel consumption column
df['fuel_idle'] = 0.0
df.loc[df['state'] == 'Idle', 'fuel_idle'] = df['liters_consumed']

# Report
n_raw = (df['state_raw'] == 'Idle_Candidate').sum()
n_final = (df['state'] == 'Idle').sum()
pct_removed = 100 * (1 - n_final / n_raw) if n_raw > 0 else 0

print(f"Criteria: speed < {SPEED_THRESHOLD * 3.6:.1f} km/h, PTO < {PTO_ACTIVE_THRESHOLD} RPM, duration ≥ {MIN_IDLE_DURATION}s")
print(f"  Slow records (candidates): {n_raw:,}")
print(f"  Confirmed Idle records   : {n_final:,}")
print(f"  {pct_removed:.1f}% reclassified as maneuvers/short pauses")

In [ ]:
# --- CELL 9: State Distribution — Time & Fuel Pie Charts ---

# Aggregate by state
state_summary = df.groupby('state').agg({
    'delta_t': 'sum',
    'liters_consumed': 'sum',
}).reset_index()

COLOR_MAP = {
    'Idle': '#d62728',     # Red
    'Working': '#2ca02c',  # Green
    'Off': '#d3d3d3',      # Light grey
}

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'domain'}, {'type': 'domain'}]],
    subplot_titles=("<b>Time Distribution</b>", "<b>Fuel Distribution</b>"),
)

# 1. Time (all states)
fig.add_trace(go.Pie(
    labels=state_summary['state'],
    values=state_summary['delta_t'],
    name="Time",
    marker_colors=[COLOR_MAP[s] for s in state_summary['state']],
    hole=0.5, sort=False, showlegend=True,
), 1, 1)

# 2. Fuel (exclude Off — negligible consumption)
fuel_summary = state_summary[state_summary['state'] != 'Off']
fig.add_trace(go.Pie(
    labels=fuel_summary['state'],
    values=fuel_summary['liters_consumed'],
    name="Fuel",
    marker_colors=[COLOR_MAP[s] for s in fuel_summary['state']],
    hole=0.5, sort=False, showlegend=False,
), 1, 2)

fig.update_layout(
    title_text="<b>Operational Efficiency:</b> Red = waste",
    title_x=0.5,
    width=800, height=450,
    margin=dict(t=80, b=30, l=40, r=40),
    legend=dict(orientation="h", yanchor="bottom", y=-0.1, xanchor="center", x=0.5),
    template=TEMPLATE,
)
fig.show()

In [ ]:
# --- CELL 10: Idle Detail — Tractor Ranking & Waste Heatmap ---

# 1. Idle hours ranking by tractor
tractor_idle = df[df['state'] == 'Idle'].groupby('tractor').agg({
    'delta_t': 'sum',
    'liters_consumed': 'sum',
}).reset_index()

tractor_idle['Idle Hours'] = tractor_idle['delta_t'] / 3600
tractor_idle = tractor_idle.sort_values('Idle Hours', ascending=True)

fig1 = px.bar(
    tractor_idle, x='Idle Hours', y='tractor', orientation='h',
    text='Idle Hours', color='liters_consumed',
    color_continuous_scale='Reds',
    title="<b>Idle Ranking:</b> Hours stationary with engine on",
    labels={'liters_consumed': 'Wasted Fuel (L)', 'tractor': 'Tractor'},
)
fig1.update_traces(texttemplate='%{text:.1f} h', textposition='outside')
fig1.update_layout(template=TEMPLATE, height=400)
fig1.show()

# 2. Waste heatmap: tractor × activity
matrix = df[df['state'] == 'Idle'].groupby(['tractor', 'activity'])['liters_consumed'].sum().reset_index()
heatmap = matrix.pivot(index='activity', columns='tractor', values='liters_consumed').fillna(0)

fig2 = px.imshow(
    heatmap,
    labels=dict(x="Tractor", y="Activity", color="Wasted Fuel (L)"),
    color_continuous_scale='Magma',
    title="<b>Waste Matrix:</b> Idle fuel by activity × tractor",
)
fig2.update_layout(height=600, template=TEMPLATE)
fig2.show()

In [ ]:
# --- CELL 11: Financial and Environmental Impact ---

DIESEL_PRICE = 1.588    # €/L
EMISSION_FACTOR = 2.64  # kg CO₂ per liter of diesel

idle_liters = df[df['state'] == 'Idle']['liters_consumed'].sum()
total_liters = df['liters_consumed'].sum()
pct_idle_diesel = (idle_liters / total_liters) * 100 if total_liters > 0 else 0

wasted_cost = idle_liters * DIESEL_PRICE
avoidable_co2_tonnes = (idle_liters * EMISSION_FACTOR) / 1000

print("=== IMPACT SUMMARY ===")
print(f"Idle share of fuel : {pct_idle_diesel:.1f}%")
print(f"Financial waste    : € {wasted_cost:,.2f}")
print(f"Avoidable CO₂      : {avoidable_co2_tonnes:.2f} tonnes")

In [ ]:
# --- CELL 12: Per-Tractor Profile — Storytelling & Sunburst ---

# 1. Aggregate by tractor × activity
profile = df.groupby(['tractor', 'activity']).agg({
    'delta_t': 'sum',
    'liters_consumed': 'sum',
}).reset_index()

# Idle-specific aggregation
profile_idle = (
    df[df['state'] == 'Idle']
    .groupby(['tractor', 'activity'])
    .agg({'delta_t': 'sum', 'liters_consumed': 'sum'})
    .reset_index()
    .rename(columns={'delta_t': 'idle_time', 'liters_consumed': 'idle_liters'})
)

profile = pd.merge(profile, profile_idle, on=['tractor', 'activity'], how='left').fillna(0)
profile['Waste (%)'] = (profile['idle_liters'] / profile['liters_consumed']) * 100
profile['Total Hours'] = profile['delta_t'] / 3600
profile['Total Fuel (L)'] = profile['liters_consumed']

# 2. Per-tractor narrative
print("=" * 80)
print("INDIVIDUAL FLEET PROFILE")
print("=" * 80)

for tractor in profile['tractor'].unique():
    td = profile[profile['tractor'] == tractor]
    
    top_diesel = td.sort_values('Total Fuel (L)', ascending=False).iloc[0]
    top_waste = td.sort_values('idle_liters', ascending=False).iloc[0]
    
    total_l = td['Total Fuel (L)'].sum()
    total_idle_l = td['idle_liters'].sum()
    pct_idle = (total_idle_l / total_l) * 100 if total_l > 0 else 0

    print(f"\n  {tractor.upper()}")
    print(f"   Total consumption  : {total_l:,.1f} L")
    print(f"   Idle waste         : {total_idle_l:,.1f} L ({pct_idle:.1f}%)")
    print(f"   Top activity       : {top_diesel['activity']} ({(top_diesel['Total Fuel (L)'] / total_l) * 100:.1f}% of fuel)")
    print(f"   Biggest bottleneck : {top_waste['activity']}")
    print(f"     ({top_waste['Waste (%)']:.1f}% of fuel in this activity was burned while idle)")

# 3. Sunburst: size = fuel consumed, color = waste %
fig = px.sunburst(
    profile,
    path=['tractor', 'activity'],
    values='Total Fuel (L)',
    color='Waste (%)',
    color_continuous_scale='RdYlGn_r',
    range_color=[0, 15],
    title="<b>Fleet Efficiency X-Ray:</b> Size = Fuel | Color = Idle Waste %",
    hover_data={'Total Hours': ':.1f', 'idle_liters': ':.1f',
                'Total Fuel (L)': ':.1f', 'Waste (%)': ':.1f'},
    labels={'idle_liters': 'Wasted Liters'},
)

fig.update_layout(
    height=700,
    margin=dict(t=50, l=0, r=0, b=0),
    template=TEMPLATE,
    font=dict(size=14),
)
fig.show()

In [ ]:
# --- CELL 13: Summary Table (Publication-Ready) ---

summary_table = []

for tractor in profile['tractor'].unique():
    td = profile[profile['tractor'] == tractor]
    
    total_l = td['liters_consumed'].sum()
    total_idle = td['idle_liters'].sum()
    pct_idle = (total_idle / total_l) * 100 if total_l > 0 else 0
    
    top_act = td.sort_values('liters_consumed', ascending=False).iloc[0]
    pct_top = (top_act['liters_consumed'] / total_l) * 100
    
    top_bottleneck = td.sort_values('idle_liters', ascending=False).iloc[0]
    
    summary_table.append({
        'Tractor': tractor,
        'Total Consumption (L)': f"{total_l:,.1f}",
        'Idle Waste': f"{total_idle:,.1f} L ({pct_idle:.1f}%)",
        'Main Activity': f"{top_act['activity']} ({pct_top:.1f}%)",
        'Biggest Bottleneck': f"{top_bottleneck['activity']} ({top_bottleneck['Waste (%)']:.1f}% idle)",
    })

df_summary = pd.DataFrame(summary_table)
display(df_summary)

In [ ]:
# --- CELL 14: Export for Dashboard App ---

df_app = df.copy()

# Drop accumulated liters — app needs instantaneous rate (L/h)
if 'liters_consumed' in df_app.columns:
    df_app = df_app.drop(columns=['liters_consumed'])

# Rename for app compatibility
df_app = df_app.rename(columns={
    'veh_speed': 'speed',
    'engine_rpm': 'engine_speed',
    'fuel_rate': 'liters_consumed',  # L/h (instantaneous rate)
})

# Keep only what the app needs
final_columns = [
    'timestamp', 'tractor', 'latitude', 'longitude',
    'speed', 'engine_speed', 'liters_consumed',
    'activity', 'state',
]
df_app = df_app[[c for c in final_columns if c in df_app.columns]]
df_app = df_app.loc[:, ~df_app.columns.duplicated()]

app_path = PROJECT_ROOT / "data" / "processed" / "telemetry_app.parquet"
df_app.to_parquet(app_path)
print(f"Saved: {app_path}")

In [ ]:
# --- CELL 15: Quick Validation — App Export ----

df_check = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "telemetry_app.parquet")
print(f"Columns: {df_check.columns.tolist()}")
print(f"Rows: {len(df_check):,}")